In [ ]:
# ============================================================================
# PEMBUAT DATASET LENGKAP & MODEL NLP UNTUK IT HELPDESK CHATBOT
# Versi Final untuk Ujian Akhir Semester
# ============================================================================

# STEP 1: INSTALL LIBRARY
!pip install Sastrawi scikit-learn pandas numpy -q

# STEP 2: IMPORT SEMUA LIBRARY
import pandas as pd
import random
import pickle
import re
import numpy as np
import json
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

print("=" * 80)
print("MEMBANGUN KNOWLEDGE BASE & MODEL NLP HELPDESK (FINAL)")
print("=" * 80)

# ============================================================================
# STEP 3: DEFINISI KNOWLEDGE BASE
# ============================================================================

knowledge_base = {
    "greeting": {
        "kategori": "Salam & Percakapan",
        "intent": "greeting",
        "masalah": [
            "halo", "hi", "pagi", "siang", "sore", "malam", "hallo",
            "assalamualaikum", "selamat pagi", "selamat siang", "selamat sore",
            "selamat malam", "apa kabar", "gimana kabar", "hows it going",
            "hey", "yo", "ada yang bisa dibantu", "butuh bantuan",
            "bisa ngomong sama admin", "ada yang bisa membantu"
        ],
        "solusi": [
            "Halo! 👋 Selamat datang di IT Help Desk SolvIT. Ada yang bisa saya bantu?",
            "Halo! Saya siap membantu masalah IT Anda. Apa yang sedang bermasalah?",
            "Selamat datang di IT Support. Saya siap melayani. Ceritakan masalahnya."
        ]
    },

    "network_internet": {
        "kategori": "Jaringan & Internet",
        "intent": "network_issue",
        "masalah": [
            "internet mati", "wifi putus", "tidak bisa konek jaringan",
            "kabel lan tidak berfungsi", "sinyal tanda seru kuning",
            "koneksi internet terputus", "wifi lemah", "kecepatan internet lambat",
            "tidak bisa terhubung ke router", "ethernet tidak terdeteksi",
            "wifi hilang", "network card tidak berfungsi", "koneksi unstable",
            "ping tinggi", "packet loss", "bandwidth terbatas", "speedtest lambat",
            "vpn tidak connect", "proxy tidak berfungsi", "firewall blokir koneksi",
            "lan cable error", "router offline", "modem tidak nyala",
            "signal lemah", "koneksi putus putus", "internet disconnect"
        ],
        "solusi": [
            "Coba lakukan langkah berikut:\n1. Periksa indikator LAN/WiFi di router\n2. Restart router selama 10 detik\n3. Lakukan 'ipconfig /release' lalu 'ipconfig /renew' di CMD\n4. Jika masih gagal, hubungi Network Support.",
            "Masalah konektivitas terdeteksi. Coba:\n1. Buka Device Manager\n2. Cari Network Adapters\n3. Klik kanan pada adapter WiFi → Update Driver\n4. Restart komputer Anda jika diminta.",
            "Cek status koneksi:\n1. Buka CMD\n2. Ketik: ping 8.8.8.8\n3. Jika Request Timed Out, periksa sambungan kabel atau indikator modem."
        ]
    },

    "printer_hardware": {
        "kategori": "Perangkat Keras - Printer",
        "intent": "printer_issue",
        "masalah": [
            "printer macet", "kertas nyangkut", "hasil print putus",
            "printer offline terus", "lampu indikator berkedip merah",
            "printer tidak terdeteksi", "driver printer error",
            "tidak bisa print dari laptop", "printing queue penuh",
            "toner habis", "printer bising aneh", "hasil print warna jelek",
            "scanner tidak berfungsi", "belum bisa scan", "hasil scan buram",
            "paper jam", "printer gangguan", "print tidak keluar"
        ],
        "solusi": [
            "Untuk printer offline:\n1. Matikan printer dan tunggu 30 detik\n2. Tekan Power ON kembali\n3. Buka Settings → Printers & Scanners\n4. Hapus dan tambahkan ulang printer.",
            "Ada kertas nyangkut:\n1. MATIKAN printer terlebih dahulu\n2. Buka semua cover panel\n3. Keluarkan kertas perlahan agar tidak sobek\n4. Tutup cover dan nyalakan ulang.",
            "Print queue penuh:\n1. Buka CMD (Admin)\n2. Ketik: net stop spooler (Enter)\n3. Ketik: net start spooler (Enter)\n4. Coba print ulang dokumen."
        ]
    },

    "system_os": {
        "kategori": "Sistem Operasi & Crash",
        "intent": "system_error",
        "masalah": [
            "layar biru", "blue screen", "bsod", "komputer restart sendiri",
            "pesan fatal error", "crash saat startup", "windows tidak bisa booting",
            "aplikasi sering crash", "memory leak", "cpu usage tinggi",
            "disk usage penuh", "system hang", "loading lambat",
            "error code 0x", "driver conflict", "update windows gagal",
            "windows error recovery", "startup repair", "system freeze"
        ],
        "solusi": [
            "Blue Screen (BSOD) - PENTING:\n1. Biarkan proses memory dump selesai\n2. Catat Stop Code yang tertera\n3. Boot ke Safe Mode (F8) jika terus berulang\n4. Hubungi IT Support dengan membawa catatan Stop Code.",
            "Komputer restart sendiri:\n1. Cek suhu CPU (overheating?) → Bersihkan fan\n2. Cek Windows Update yang pending\n3. Uninstall software yang baru dipasang hari ini.",
            "Disk usage 100% / Drive C: penuh:\n1. Buka File Explorer → This PC\n2. Klik kanan C: Drive → Properties\n3. Buka Disk Cleanup tool\n4. Bersihkan file Temporary."
        ]
    },

    "software_issue": {
        "kategori": "Software & Aplikasi",
        "intent": "software_problem",
        "masalah": [
            "aplikasi tidak bisa dibuka", "error saat install software",
            "lisensi software expired", "software illegal", "crack software",
            "aktivasi office gagal", "microsoft office tidak berfungsi",
            "browser tidak bisa buka website", "chrome error", "firefox tidak merespons",
            "file tidak bisa dibuka", "format file tidak didukung",
            "software lama sudah tidak support", "ada bug di aplikasi",
            "program error", "aplikasi loading lama", "runtime error"
        ],
        "solusi": [
            "Aplikasi tidak bisa dibuka:\n1. Cek Task Manager (Ctrl+Shift+Esc), kill process yang nyangkut\n2. Klik kanan aplikasi → Run as Administrator\n3. Reinstall jika file corrupt.",
            "Aktivasi Office bermasalah:\n1. Buka Word/Excel → File → Account → Sign out\n2. Sign in kembali dengan akun email kantor yang valid\n3. Tunggu 5 menit untuk proses sinkronisasi lisensi.",
            "Browser tidak bisa buka web:\n1. Clear cache (Ctrl+Shift+Delete)\n2. Coba mode Incognito (Ctrl+Shift+N)\n3. Cek apakah alamat web terblokir oleh Firewall kantor."
        ]
    },

    "security_issue": {
        "kategori": "Security & Access Control",
        "intent": "security_problem",
        "masalah": [
            "lupa password", "account terkunci", "tidak bisa login",
            "virus terdeteksi", "malware infection", "ransomware attack",
            "secure boot off", "windows defender off", "firewall disabled",
            "suspicious activity", "antivirus error", "security warning"
        ],
        "solusi": [
            "Lupa password Windows:\n1. Gunakan opsi 'Reset password' di layar login\n2. Jika terkunci, hubungi IT Help Desk dengan ID Karyawan untuk proses reset dari Active Directory.",
            "Virus / Malware terdeteksi:\n1. JANGAN buka file mencurigakan\n2. Putuskan koneksi internet/LAN\n3. Lakukan Full Scan menggunakan Windows Defender\n4. Laporkan ke IT Security secepatnya.",
            "Account terkunci:\n1. Jangan terus mencoba login (menghindari lockout permanen)\n2. Tunggu 15 menit, pastikan Caps Lock mati\n3. Hubungi admin untuk unlock akun."
        ]
    }
}

# ============================================================================
# STEP 4: AUGMENTASI DATA
# ============================================================================

print("\n📊 Melakukan Data Augmentation...")

def augment_pertanyaan(pertanyaan):
    variasi = [
        pertanyaan,
        pertanyaan.upper(),
        pertanyaan.title(),
        f"saya punya masalah: {pertanyaan}",
        f"bantuan! {pertanyaan}",
        f"tolong, {pertanyaan}",
        f"ada yang bisa bantu? {pertanyaan}",
        f"gimana ya {pertanyaan}?",
        f"error: {pertanyaan}",
        f"{pertanyaan} nih, bisa dibantu?"
    ]
    return variasi

data_sintetis = []

for intent_key, intent_data in knowledge_base.items():
    kategori = intent_data["kategori"]
    masalah_list = intent_data["masalah"]
    solusi_list = intent_data["solusi"]
    intent = intent_data["intent"]

    for masalah in masalah_list:
        for pertanyaan_aug in augment_pertanyaan(masalah):
            solusi = random.choice(solusi_list)
            data_sintetis.append({
                "pertanyaan": pertanyaan_aug,
                "intent": intent,
                "kategori": kategori,
                "solusi": solusi
            })

df = pd.DataFrame(data_sintetis)
df = df.drop_duplicates(subset=['pertanyaan']).reset_index(drop=True)
print(f"✅ Total dataset: {len(df)} baris")

# ============================================================================
# STEP 5: PREPROCESSING TEXT (SASTRAWI STEMMING)
# ============================================================================

print("\n🔧 Preprocessing text dengan Sastrawi stemming (Tunggu Sebentar)...")

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def bersihkan_teks(teks):
    teks = str(teks).lower()
    teks = re.sub(r'[^\w\s]', '', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    words = teks.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    return ' '.join(stemmed_words)

df['pertanyaan_bersih'] = df['pertanyaan'].apply(bersihkan_teks)
print("✅ Preprocessing selesai")

# ============================================================================
# STEP 6: FEATURE EXTRACTION (TF-IDF)
# ============================================================================

print("\n📈 Ekstraksi fitur dengan TF-IDF...")

# REVISI: Dihapus parameter stop_words='english' karena menggunakan bahasa Indonesia
vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=1,
    max_df=0.9,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(df['pertanyaan_bersih'])
print(f"✅ TF-IDF matrix shape: {tfidf_matrix.shape}")

# ============================================================================
# STEP 7: EVALUASI & TRAINING (REVISI UAS WAJIB)
# ============================================================================

print("\n🤖 Membagi Data & Evaluasi Model (SYARAT UAS)...")

# 1. Membagi Data Latih (80%) dan Data Uji (20%)
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, df['intent'], test_size=0.2, random_state=42)

# 2. Melatih Classifier Naive Bayes
clf = MultinomialNB()
clf.fit(X_train, y_train)

# 3. Melakukan Prediksi pada Data Uji
y_pred = clf.predict(X_test)

# 4. Mencetak 5 Metrik Evaluasi Wajib
print(f"\nAkurasi   : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Precision : {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")

print("\nLaporan Klasifikasi (Classification Report):")
print(classification_report(y_test, y_pred, zero_division=0))

print("Matriks Kebingungan (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred))

# 5. Latih ulang menggunakan SELURUH data agar model akhir maksimal untuk aplikasi
print("\nMelatih ulang model final dengan seluruh dataset...")
clf.fit(tfidf_matrix, df['intent'])

# ============================================================================
# STEP 8: PENYIMPANAN FILE MODEL
# ============================================================================

print("\n💾 Menyimpan semua file output...")

df.to_csv("dataset_ithelpdesk_final.csv", index=False, encoding='utf-8')
with open("knowledge_base.json", "w", encoding='utf-8') as f:
    json.dump(knowledge_base, f, ensure_ascii=False, indent=2)
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
with open("tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)
with open("intent_classifier.pkl", "wb") as f:
    pickle.dump(clf, f)

print("✅ Semua file model telah disimpan dengan aman.")

# ============================================================================
# STEP 9: TESTING SIMULASI (SELESAI)
# ============================================================================

print("\n" + "=" * 80)
print("🎉 PROSES PELATIHAN AI HELPDESK SELESAI!")
print("=" * 80)

def simulasi_chat(user_input):
    clean_input = bersihkan_teks(user_input)
    vec_input = vectorizer.transform([clean_input])
    intent_prediksi = clf.predict(vec_input)[0]

    skor_kemiripan = cosine_similarity(vec_input, tfidf_matrix)[0]
    best_idx = skor_kemiripan.argmax()
    skor_tertinggi = skor_kemiripan[best_idx]

    if skor_tertinggi < 0.15:
        return "Mohon maaf, saya belum memahami masalah tersebut."
    return df.iloc[best_idx]['solusi']

print("\nSimulasi Tes Cepat:")
print("Tanya: 'Laptop layarnya biru nih pak'")
print("Jawab AI:", simulasi_chat("Laptop layarnya biru nih pak"))

MEMBANGUN KNOWLEDGE BASE & MODEL NLP HELPDESK (FINAL)

📊 Melakukan Data Augmentation...
✅ Total dataset: 1130 baris

🔧 Preprocessing text dengan Sastrawi stemming (Tunggu Sebentar)...
✅ Preprocessing selesai

📈 Ekstraksi fitur dengan TF-IDF...
✅ TF-IDF matrix shape: (1130, 860)

🤖 Membagi Data & Evaluasi Model (SYARAT UAS)...

Akurasi   : 100.00%
Precision : 1.0000
Recall    : 1.0000
F1-Score  : 1.0000

Laporan Klasifikasi (Classification Report):
                  precision    recall  f1-score   support

        greeting       1.00      1.00      1.00        45
   network_issue       1.00      1.00      1.00        53
   printer_issue       1.00      1.00      1.00        37
security_problem       1.00      1.00      1.00        20
software_problem       1.00      1.00      1.00        37
    system_error       1.00      1.00      1.00        34

        accuracy                           1.00       226
       macro avg       1.00      1.00      1.00       226
    weighted avg       1